In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df['Class'].value_counts(normalize=True)

In [ ]:
df[['Time', 'Amount']].describe()

In [ ]:
df.isnull().sum()

In [ ]:
hour = (df['Time'] % 86400) / 3600  # 0-24
df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df = df.drop(columns=['Time'])

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Split remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [ ]:
print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
print("Train Fraud Ratio:")
print(y_train.value_counts(normalize=True))

print("\nValidation Fraud Ratio:")
print(y_val.value_counts(normalize=True))

print("\nTest Fraud Ratio:")
print(y_test.value_counts(normalize=True))

In [ ]:
print("V1 Statistics")
print(X_train['V1'].describe())

print("\nAmount Statistics")
print(X_train['Amount'].describe())

In [ ]:
from sklearn.preprocessing import RobustScaler
import pandas as pd

scaler = RobustScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
print(X_train_scaled['V1'].describe())
print(X_train_scaled['Amount'].describe())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_val_scaled)

# Probability of fraud (Class = 1)
y_prob = model.predict_proba(X_val_scaled)[:, 1]

In [ ]:
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
pr_auc = average_precision_score(y_val, y_prob)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

In [ ]:
print("X_train shape:", X_train.shape)
print("Number of features:", X_train.shape[1])